# 01 · Carga Delta — Landing → Bronze
### Proyecto ETL FIFA 21 · Jorge Amat · David Plaza

**Requisito previo:** haber ejecutado `00_setup_catalogo` (catálogo, esquemas y volumen ya creados) y tener el CSV subido en `/Volumes/fifa_catalog/bronze/landing/`.

**Qué hace este notebook:**
1. Lee el CSV del volumen con `multiLine=True` (el dataset trae saltos de línea embebidos en algunos campos).
2. Comprueba nulos por columna, para verificar que el CSV se leyó bien.
3. Limpia los nombres de columna (espacios, símbolos, el caso especial de `Team & Contract` → `team`).
4. Escribe **directamente** la tabla Delta de Bronze: `fifa_catalog.bronze.fifa_21_delta`.

**Siguiente notebook:** `transformaciones.ipynb`

## 1. Leer el CSV bien (multiLine)

`multiLine=True` + `escape='"'` le dicen a Spark que un campo entrecomillado puede contener saltos de línea, y que no debe cortar la fila ahí.

In [0]:
CSV_PATH = "/Volumes/fifa_catalog/bronze/landing/fifa21_raw_data.csv"

df_bronze = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print("Filas leídas:", df_bronze.count())
print("Columnas:", len(df_bronze.columns))
display(df_bronze.limit(20))

Filas leídas: 18979
Columnas: 77


photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,↓OVA,POT,Team & Contract,ID,Height,Weight,foot,BOV,BP,Growth,Joined,Loan Date End,Value,Wage,Release Clause,Attacking,Crossing,Finishing,Heading Accuracy,Short Passing,Volleys,Skill,Dribbling,Curve,FK Accuracy,Long Passing,Ball Control,Movement,Acceleration,Sprint Speed,Agility,Reactions,Balance,Power,Shot Power,Jumping,Stamina,Strength,Long Shots,Mentality,Aggression,Interceptions,Positioning,Vision,Penalties,Composure,Defending,Marking,Standing Tackle,Sliding Tackle,Goalkeeping,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes,Total Stats,Base Stats,W/F,SM,A/W,D/W,IR,PAC,SHO,PAS,DRI,DEF,PHY,Hits
https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,RW ST CF,L. Messi,33,93,93,FC Barcelona 2004 ~ 2021,158023,"5'7""",159lbs,Left,93,RW,0,"Jul 1, 2004",N/A,€67.5M,€560K,€138.4M,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4 ★,4★,Medium,Low,5 ★,85,92,91,95,38,65,372
https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,ST LW,Cristiano Ronaldo,35,92,92,Juventus 2018 ~ 2022,20801,"6'2""",183lbs,Right,92,ST,0,"Jul 10, 2018",N/A,€46M,€220K,€75.9M,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4 ★,5★,High,Low,5 ★,89,93,81,89,35,77,344
https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,Atlético Madrid 2014 ~ 2023,200389,"6'2""",192lbs,Right,91,GK,2,"Jul 16, 2014",N/A,€75M,€125K,€159.4M,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3 ★,1★,Medium,Medium,3 ★,87,92,78,90,52,90,86
https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,CAM CM,K. De Bruyne,29,91,91,Manchester City 2015 ~ 2023,192985,"5'11""",154lbs,Right,91,CAM,0,"Aug 30, 2015",N/A,€87M,€370K,€161M,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5 ★,4★,High,High,4 ★,76,86,93,88,64,78,163
https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,LW CAM,Neymar Jr,28,91,91,Paris Saint-Germain 2017 ~ 2022,190871,"5'9""",150lbs,Right,91,LW,0,"Aug 3, 2017",N/A,€90M,€270K,€166.5M,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5 ★,5★,High,Medium,5 ★,91,85,86,94,36,59,273
https://cdn.sofifa.com/players/188/545/21_60.png,Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,ST,R. Lewandowski,31,91,91,FC Bayern München 2014 ~ 2023,188545,"6'0""",176lbs,Right,91,ST,0,"Jul 1, 2014",N/A,€80M,€240K,€132M,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4 ★,4★,High,Medium,4 ★,78,91,78,85,43,82,182
https://cdn.sofifa.com/players/231/747/21_60.png,Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,ST LW RW,K. Mbappé,21,90,95,Paris Saint-Germain 2018 ~ 2022,231747,"5'10""",161lbs,Right,91,ST,5,"Jul 1, 2018",N/A,€105.5M,€160K,€203.1M,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4 ★,5★,High,Low,3 ★,96,86,78,91,39,76,646
https://cdn.sofifa.com/players/212/831/21_60.png,Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses-becker/210005/,Brazil,GK,Alisson,27,90,91,Liverpool 2018 ~ 2024,212831,"6'3""",201lbs,Right,90,GK,1,"Jul 19, 2018",N

In [0]:
# Comprobación rápida de nulos por columna: si el CSV se ha leído bien,
# no deberían aparecer nulos masivos y aleatorios en columnas que sabemos que están casi siempre informadas
# (por ejemplo longname, age, nationality no deberían tener apenas nulos)
from pyspark.sql import functions as f

df_bronze.select([
    f.sum(f.col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns
]).show(vertical=True, truncate=False)

-RECORD 0---------------
 photoUrl         | 0   
 LongName         | 0   
 playerUrl        | 0   
 Nationality      | 0   
 Positions        | 0   
 Name             | 0   
 Age              | 0   
 ↓OVA             | 0   
 POT              | 0   
 Team & Contract  | 0   
 ID               | 0   
 Height           | 0   
 Weight           | 0   
 foot             | 0   
 BOV              | 0   
 BP               | 0   
 Growth           | 0   
 Joined           | 0   
 Loan Date End    | 0   
 Value            | 0   
 Wage             | 0   
 Release Clause   | 0   
 Attacking        | 0   
 Crossing         | 0   
 Finishing        | 0   
 Heading Accuracy | 0   
 Short Passing    | 0   
 Volleys          | 0   
 Skill            | 0   
 Dribbling        | 0   
 Curve            | 0   
 FK Accuracy      | 0   
 Long Passing     | 0   
 Ball Control     | 0   
 Movement         | 0   
 Acceleration     | 0   
 Sprint Speed     | 0   
 Agility          | 0   
 Reactions        | 0   


## 2. Limpiar nombres de columna

Delta no admite espacios ni ciertos simbolos (`&`, `(`, `)`, etc.) en nombres de columna, y el CSV original los trae tal cual (por ejemplo 'Team & Contract', 'Loan Date End'). Los normalizamos antes de escribir la tabla.

In [0]:
import re

def limpiar_nombre_columna(c):
    c = c.strip().lower().replace(" ", "_")
    c = re.sub(r"[^a-zA-Z0-9_]", "_", c)
    return c

for c in df_bronze.columns:
    nuevo_nombre = limpiar_nombre_columna(c)
    if nuevo_nombre != c:
        df_bronze = df_bronze.withColumnRenamed(c, nuevo_nombre)

# Caso especial: "Team & Contract" trae club y contrato combinados en el CSV original.
# La limpieza generica la deja como "team___contract", pero necesitamos que se llame "team".
for c in df_bronze.columns:
    if "team" in c and "contract" in c:
        df_bronze = df_bronze.withColumnRenamed(c, "team")
        break

print("Columnas renombradas:")
print(df_bronze.columns)

Columnas renombradas:
['photourl', 'longname', 'playerurl', 'nationality', 'positions', 'name', 'age', '_ova', 'pot', 'team', 'id', 'height', 'weight', 'foot', 'bov', 'bp', 'growth', 'joined', 'loan_date_end', 'value', 'wage', 'release_clause', 'attacking', 'crossing', 'finishing', 'heading_accuracy', 'short_passing', 'volleys', 'skill', 'dribbling', 'curve', 'fk_accuracy', 'long_passing', 'ball_control', 'movement', 'acceleration', 'sprint_speed', 'agility', 'reactions', 'balance', 'power', 'shot_power', 'jumping', 'stamina', 'strength', 'long_shots', 'mentality', 'aggression', 'interceptions', 'positioning', 'vision', 'penalties', 'composure', 'defending', 'marking', 'standing_tackle', 'sliding_tackle', 'goalkeeping', 'gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning', 'gk_reflexes', 'total_stats', 'base_stats', 'w_f', 'sm', 'a_w', 'd_w', 'ir', 'pac', 'sho', 'pas', 'dri', 'def', 'phy', 'hits']


## 3. Escribir la tabla Delta de Bronze

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "fifa_catalog.bronze.fifa_21_delta"
)
print("✅ Tabla Bronze escrita correctamente: fifa_catalog.bronze.fifa_21_delta")

✅ Tabla Bronze escrita correctamente: fifa_catalog.bronze.fifa_21_delta
